# Stage 2 Notebook 24 - Exp2S Bezier-curve query head

**Why this exists.** Across Exp2P/Q/R the query-style cls converged in 10 epochs (val_lane_f1=0.65) but geometry stayed at matched_iou ~0.13 because predicting `start_y/start_x/theta/length + 72 row offsets` (76 dof per lane) is too much for K=12 random-init queries to learn in 10 epochs.

**Bezier-curve representation** (BezierLaneNet, CVPR 2022) replaces those 76 dof with **4 cubic Bezier control points (8 dof per lane)**. An 8x reduction in output dimensionality. Cubic Bezier exactly represents straight and gentle-curve lanes, which dominate BDD. The compact parameterization should let geometry converge much faster.

Implementation in `BezierLaneQueryHead`:
- K=12 queries through the same 3-layer transformer decoder used in Exp2P/R.
- Per-query output: cls (1d) + 4 control points (8d) = 9 dof.
- 72-point coord_pred is sampled deterministically from the Bezier curve at evenly-spaced t in [0, 1]. This keeps FusionLaneLoss + LaneF1DecodedMetric working unchanged.
- Lane params (start_y, start_x, theta, length) and row offsets are derived analytically from the sampled curve so the existing reg/xytl/smooth losses still apply.

Reference: BezierLaneNet (Feng et al. 2022, CVPR). Cubic Bezier from least-squares fit to GT points (helper `_gt_lanes_to_bezier_targets` in lane_head.py is exposed for an optional control-point regression loss in a follow-up if needed).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp19_rmt_gca_bezier_query_joint_smoke.log
OK exp19_rmt_gca_bezier_query_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=1.5364 det_loss=3.6681 grad_cos=-0.2965 lambda_lane=0.0623
  gate_stats={'gate/det_mean': 0.5006921291351318, 'gate/lane_mean': 0.5006141066551208, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp19_rmt_gca_bezier_query_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp19_rmt_gca_bezier_query_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve

0

## What to watch in Exp2S training

Reference Exp2P (queries, no Bezier): val_lane_f1=0.65, matched_iou=0.13, decoded_f1=0.026.

Pass criteria at epoch 10:

- **`val/matched_line_iou >= 0.30`**: 8x smaller output space should let geometry converge much faster than Exp2P's 0.13.
- **`val/lane_exist_best_f1 >= 0.55`**: cls task is unchanged from Exp2P; should still hit ~0.65.
- **`val/lane/decoded_f1 >= 0.10`**, ideally `>= 0.15`: 4-6x our current best.
- **`val/lane/decoded_oracle_f1 >= 0.18`**: better geometry should lift the oracle ceiling proportionally.

Failure signals -> next ablation:

- Geometry doesn't recover (matched_iou < 0.20): cubic Bezier insufficient for BDD lanes (some have 4+ inflection points). Try quintic Bezier (6 control points, 12 dof) or cap to fixed-y row anchors with learnable Bezier-x.
- Cls regresses: parameterization shift broke the cls/geometry balance; raise `w_cls: 4.0 -> 6.0`.
- Both fail: representation isn't the bottleneck after all; the impasse comes from training data/duration.